# Day 3 · Session 1 · Notebook 01B  
# Structured Output with OpenAI Agents SDK

## Objective

In the previous notebook, we created a basic agent that returned normal text.

In this notebook, we will make the agent return **structured data**.

This is important because in enterprise applications, we do not always want paragraphs.

Sometimes we want:

- JSON
- Tables
- Scores
- Categories
- Action items
- Form-like output

## Teaching message

A normal LLM answer is good for humans.  
A structured output is good for software.

## Step 1 — Import Libraries

We need:

- `Agent` and `Runner` from OpenAI Agents SDK
- `pydantic.BaseModel` to define the structure of the output
- `dotenv` to load the API key

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from agents import Agent, Runner

print("Libraries imported successfully.")

## Step 2 — Load API Key from Root `.env`

Your `.env` file is kept in the parent project folder.

This code searches upward from the current folder until it finds `.env`.

So it works even when you run this notebook from inside the Day 3 folder.

In [ ]:
def load_project_env():
    current = Path.cwd().resolve()

    for folder in [current] + list(current.parents):
        env_path = folder / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            print(f"Loaded .env from: {env_path}")
            return env_path

    raise FileNotFoundError("Could not find .env file in current or parent folders.")

load_project_env()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

print("OPENAI_API_KEY loaded successfully.")

## Step 3 — Why Structured Output?

Let us first understand the problem.

If we ask an LLM to explain a concept, it may return a beautiful paragraph.

But what if our software needs to store:

- topic
- difficulty
- explanation
- examples
- quiz question

A paragraph is hard to parse.

Structured output solves this.

## Step 4 — Define the Output Structure

We will create a Pydantic model.

Think of this as a form that the AI must fill.

The AI is not free to return any random format. It must follow this structure.

In [ ]:
class TeachingExplanation(BaseModel):
    topic: str = Field(description="The topic being explained")
    difficulty_level: str = Field(description="Beginner, Intermediate, or Advanced")
    short_explanation: str = Field(description="A simple explanation in 2-3 sentences")
    classroom_example: str = Field(description="A relatable classroom or college example")
    quiz_question: str = Field(description="One question to check understanding")

print("Structured output model created.")

## Step 5 — Create an Agent with Structured Output

Notice the important parameter:

```python
output_type=TeachingExplanation
```

This tells the SDK:

> Do not return plain text. Return data matching this Pydantic model.

In [ ]:
teaching_agent = Agent(
    name="Structured Teaching Assistant",
    instructions="""
You are a helpful teaching assistant for engineering faculty.

Explain AI and ML concepts clearly.
Use simple examples suitable for Indian engineering college classrooms.
Always return the response in the requested structured format.
""",
    model="gpt-4o-mini",
    output_type=TeachingExplanation,
)

print("Structured output agent created.")

## Step 6 — Run the Agent

In VS Code notebooks, we use:

```python
await Runner.run(...)
```

This works well because notebooks already have an async event loop.

In [ ]:
result = await Runner.run(
    teaching_agent,
    "Explain overfitting in machine learning."
)

print(result.final_output)

## Step 7 — Access Individual Fields

The output is not just text.

It is a Python object.

That means we can access each field separately.

In [ ]:
output = result.final_output

print("Topic:", output.topic)
print("Difficulty:", output.difficulty_level)
print("\nExplanation:")
print(output.short_explanation)
print("\nExample:")
print(output.classroom_example)
print("\nQuiz:")
print(output.quiz_question)

## Step 8 — Convert to Dictionary

In real applications, structured output can be stored in:

- database
- API response
- CSV
- dashboard
- LMS system

In [ ]:
output_dict = output.model_dump()
output_dict

## Step 9 — Another Example: Lesson Plan Generator

Now let us create a slightly richer structure.

This is useful for faculty because the AI can generate a mini teaching plan.

In [ ]:
class MiniLessonPlan(BaseModel):
    subject: str
    topic: str
    learning_objectives: list[str]
    explanation_flow: list[str]
    demo_idea: str
    assessment_question: str
    common_misconception: str

lesson_agent = Agent(
    name="Mini Lesson Plan Generator",
    instructions="""
You help engineering faculty prepare short classroom teaching plans.
Keep the content practical and classroom-friendly.
""",
    model="gpt-4o-mini",
    output_type=MiniLessonPlan,
)

print("Mini lesson plan agent created.")

In [ ]:
lesson_result = await Runner.run(
    lesson_agent,
    "Create a 15-minute lesson plan for Retrieval Augmented Generation."
)

lesson = lesson_result.final_output
lesson

## Step 10 — Print the Lesson Plan Nicely

In [ ]:
print("Subject:", lesson.subject)
print("Topic:", lesson.topic)

print("\nLearning Objectives:")
for obj in lesson.learning_objectives:
    print("-", obj)

print("\nExplanation Flow:")
for step in lesson.explanation_flow:
    print("-", step)

print("\nDemo Idea:")
print(lesson.demo_idea)

print("\nAssessment Question:")
print(lesson.assessment_question)

print("\nCommon Misconception:")
print(lesson.common_misconception)

## Step 11 — Classroom Demo Idea

Ask the faculty:

> What happens if we do not use structured output?

Then explain:

Without structured output:
- every answer may look different
- software cannot reliably parse it
- downstream automation becomes difficult

With structured output:
- predictable format
- easy to validate
- easy to store
- easy to connect with applications

## Step 12 — Exercise

Modify the `MiniLessonPlan` model.

Add these fields:

```python
recommended_duration: int
prerequisites: list[str]
lab_activity: str
```

Then ask the agent to prepare a lesson plan for:

> "Prompt Engineering for First Year Engineering Students"

In [ ]:
# Exercise starter

class ExtendedLessonPlan(BaseModel):
    subject: str
    topic: str
    learning_objectives: list[str]
    recommended_duration: int
    prerequisites: list[str]
    explanation_flow: list[str]
    lab_activity: str
    assessment_question: str

extended_agent = Agent(
    name="Extended Lesson Plan Generator",
    instructions="""
Create a structured lesson plan for engineering faculty.
Use simple language and practical classroom activities.
""",
    model="gpt-4o-mini",
    output_type=ExtendedLessonPlan,
)

extended_result = await Runner.run(
    extended_agent,
    "Create a lesson plan for Prompt Engineering for First Year Engineering Students."
)

extended_result.final_output

## Key Takeaways

1. Structured output converts LLM responses into predictable data.
2. Pydantic models define the shape of the answer.
3. This is extremely useful for enterprise AI applications.
4. It helps connect LLMs with databases, dashboards, APIs, and workflows.
5. For faculty, it can generate structured lesson plans, quiz banks, rubrics, and evaluation forms.

## Transition to Next Notebook

Now that the agent can return structured data, the next step is:

> Can the agent use a Python function as a tool?

That is Notebook 01C — Function Tools.